# Indie Games Sales Prediction — Baseline ML Pipeline

**Goal:** clean the Steam revenue dataset, preprocess categorical/numerical predictors without data leakage, and train a Linear Regression baseline for `CopiesSold`.

**Dataset:** [Top 1500 Games on Steam by Revenue (09-09-2024)](https://www.kaggle.com/datasets/alicemtopcu/top-1500-games-on-steam-by-revenue-09-09-2024)

## How to run
1. Download `indie_games.csv` from Kaggle.
2. Upload it to this Colab notebook when prompted, or place it in the notebook's working directory.
3. Run all cells from top to bottom.

> This notebook intentionally inspects the actual CSV before making dataset-specific assumptions. The exact columns and metrics depend on the file you upload. Do not report placeholder results as model results.


In [ ]:
# Imports
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)


## 1. Load the dataset
In Google Colab, upload `indie_games.csv` when prompted. If you have already placed it in the current directory, the fallback path works.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next((name for name in uploaded if name.lower().endswith(".csv")), None)
    if csv_path is None:
        raise FileNotFoundError("Please upload a CSV file.")
except ImportError:
    csv_path = "indie_games.csv"

df_raw = pd.read_csv(csv_path)
print("Raw shape:", df_raw.shape)
display(df_raw.head())
print("\\nColumn names and data types:")
display(pd.DataFrame({"column": df_raw.columns, "dtype": df_raw.dtypes.astype(str).values}))


## 2. Data-quality audit
Review missingness, duplicate rows, unique values, and suspicious high-cardinality columns before deciding what to remove. These diagnostics are evidence for the cleaning decisions rather than assumptions about the dataset.


In [ ]:
df = df_raw.copy()
print("Duplicate rows:", int(df.duplicated().sum()))
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
    "dtype": df.dtypes.astype(str)
}).sort_values(["missing_percent", "unique_values"], ascending=[False, False])
display(missing_report)

print("\\nSample values by column:")
for col in df.columns:
    print(f"\\n{col} ({df[col].dtype}) — unique={df[col].nunique(dropna=True)}")
    print(df[col].dropna().astype(str).head(5).tolist())


## 3. Cleaning and target validation
The target must be `CopiesSold`. This cell removes exact duplicate rows, normalizes column names, parses common numeric formatting, and reports target problems. Rows with missing or non-positive target values are excluded because a supervised model cannot learn from an absent or invalid sales target. The parser handles commas and common currency symbols; inspect conversion failures before proceeding.


In [ ]:
# Normalize column labels for easier access, while preserving meaning.
df.columns = [re.sub(r"[^0-9a-zA-Z]+", "_", str(c).strip()).strip("_") for c in df.columns]
df = df.drop_duplicates().copy()

if "CopiesSold" not in df.columns:
    candidates = [c for c in df.columns if c.lower().replace("_", "") == "copiessold"]
    if candidates:
        df = df.rename(columns={candidates[0]: "CopiesSold"})
    else:
        raise KeyError(f"Target 'CopiesSold' not found. Available columns: {df.columns.tolist()}")

def parse_numeric(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = (series.astype("string")
               .str.replace(",", "", regex=False)
               .str.replace(r"[$£€]", "", regex=True)
               .str.replace(r"(?i)\\b(copies|units|sold)\\b", "", regex=True)
               .str.strip())
    return pd.to_numeric(cleaned, errors="coerce")

# Convert target; keep a report so unexpected formats are visible.
target_before = df["CopiesSold"].copy()
df["CopiesSold"] = parse_numeric(df["CopiesSold"])
print("Target values converted to NaN:", int(df["CopiesSold"].isna().sum() - target_before.isna().sum()))
print("Missing target:", int(df["CopiesSold"].isna().sum()))
print("Non-positive target:", int((df["CopiesSold"] <= 0).sum()))

df = df.loc[df["CopiesSold"].notna() & (df["CopiesSold"] > 0)].copy()
print("Shape after duplicate/target cleaning:", df.shape)
display(df.head())


## 4. Feature selection and category cleanup
IDs, URLs, descriptions, and other free-text fields are often unsuitable for a simple baseline unless deliberately engineered. We use a conservative, name-based drop list only when those columns exist; inspect the printed list and adjust it if an apparently irrelevant field contains useful signal. Categorical strings are trimmed and blank strings become missing. Near-duplicate category spellings should be harmonized only when they are clearly equivalent.


In [ ]:
# Candidate identifiers / high-noise text fields. Only existing columns are dropped.
drop_exact = {
    "appid", "app_id", "steamid", "steam_id", "url", "game_url",
    "website", "description", "about_the_game", "header_image",
    "capsule_image", "icon", "thumbnail", "notes"
}
drop_cols = [c for c in df.columns if c.lower() in drop_exact]
print("Columns proposed for removal:", drop_cols)

# Drop constant columns and obvious identifiers (unique in every row) after excluding target.
constant_cols = [c for c in df.columns if c != "CopiesSold" and df[c].nunique(dropna=False) <= 1]
print("Constant columns proposed for removal:", constant_cols)
drop_cols = sorted(set(drop_cols + constant_cols))
df = df.drop(columns=drop_cols, errors="ignore")

# Normalize categorical strings; don't apply fuzzy replacement that could merge distinct labels.
for col in df.select_dtypes(include=["object", "string", "category"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Identify high-cardinality columns for manual review rather than silently encoding them.
high_cardinality = [
    c for c in df.select_dtypes(include=["object", "string", "category"]).columns
    if df[c].nunique(dropna=True) > min(100, max(30, int(0.2 * len(df))))
]
print("High-cardinality text/categorical columns to review:", high_cardinality)
print("Current columns:", df.columns.tolist())


## 5. Inspect outliers
Sales and revenue datasets are commonly right-skewed. We flag outliers with the IQR rule for numeric columns, but do **not** automatically delete high-sales games: those may be legitimate and important observations. The baseline uses `log1p(CopiesSold)` as the regression target to reduce skew, then converts predictions back to the original scale for evaluation. This keeps the task's target meaning intact while making a linear baseline less dominated by the largest titles.


In [ ]:
numeric_for_audit = df.select_dtypes(include=np.number).columns.tolist()
outlier_rows = []
for col in numeric_for_audit:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((df[col] < low) | (df[col] > high)).sum())
    outlier_rows.append({"column": col, "lower_IQR_bound": low, "upper_IQR_bound": high,
                         "outlier_count": count, "outlier_percent": round(100*count/len(df), 2)})
display(pd.DataFrame(outlier_rows).sort_values("outlier_percent", ascending=False))


## 6. Prepare features and train-test split
The target is excluded from `X`. Any columns that directly reveal the target (for example, a duplicate sales estimate) should be removed after checking the data dictionary. A random split is used for a reproducible baseline. All imputing, encoding, and scaling are fitted on the training partition only through a scikit-learn pipeline, preventing test-set leakage.


In [ ]:
# Exclude target and any exact duplicate target-like fields if present.
leakage_names = {"copies_sold", "copiessold_estimate", "estimated_copies_sold"}
leakage_cols = [c for c in df.columns if c != "CopiesSold" and c.lower() in leakage_names]
if leakage_cols:
    print("Dropping possible target-leakage columns:", leakage_cols)
    df = df.drop(columns=leakage_cols)

X = df.drop(columns=["CopiesSold"]).copy()
y = df["CopiesSold"].astype(float)

# Remove columns with excessive missingness (>80%) to avoid mostly-imputed noise.
high_missing_cols = X.columns[X.isna().mean() > 0.80].tolist()
print("Dropping features with >80% missing values:", high_missing_cols)
X = X.drop(columns=high_missing_cols)

# Treat booleans as categorical. Numeric-looking object columns are parsed only if nearly all
# non-null values can be converted, to avoid corrupting genuine category labels.
for col in X.select_dtypes(include=["object", "string"]).columns:
    parsed = parse_numeric(X[col])
    non_null = X[col].notna().sum()
    if non_null > 0 and parsed.notna().sum() / non_null >= 0.95:
        X[col] = parsed

bool_cols = X.select_dtypes(include=["bool"]).columns.tolist()
for col in bool_cols:
    X[col] = X[col].astype("string")

# Exclude extremely high-cardinality free text from a simple baseline.
cat_cols_initial = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
high_card_cols = [c for c in cat_cols_initial if X[c].nunique(dropna=True) > min(100, max(30, int(0.2*len(X))))]
print("Dropping high-cardinality text columns (review first):", high_card_cols)
X = X.drop(columns=high_card_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)


## 7. One-Hot Encoding, Label Encoding, and scaling
One-Hot Encoding is used for nominal categories because it does not impose a fake order. Label Encoding is generally intended for target labels in classification; applying arbitrary integer codes to nominal predictors can mislead linear regression. Therefore, this pipeline demonstrates `LabelEncoder` only as an optional example for a truly ordinal feature (after the user explicitly defines its order), while the actual model uses One-Hot Encoding. `StandardScaler` is selected for numeric predictors because it centers features and scales them to unit variance; `MinMaxScaler` is compared below. The scaler is fitted only on training data.


In [ ]:
# Optional LabelEncoder demonstration: only use when a column has a genuine ordinal order.
# Example template (leave commented unless you know the order is meaningful):
# ordered_levels = ["Low", "Medium", "High"]
# le = LabelEncoder()
# le.fit(ordered_levels)
# df["ExampleOrdinal_encoded"] = le.transform(df["ExampleOrdinal"].astype(str))

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder="drop", verbose_feature_names_out=False)

# Fit preprocessing on training data only, then transform train and test.
X_train_ready_array = preprocessor.fit_transform(X_train)
X_test_ready_array = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
X_train_ready = pd.DataFrame(X_train_ready_array, columns=feature_names, index=X_train.index)
X_test_ready = pd.DataFrame(X_test_ready_array, columns=feature_names, index=X_test.index)

print("Final machine-learning-ready feature matrix (training partition) BEFORE model training:")
display(X_train_ready.head())
print("Ready train matrix:", X_train_ready.shape)
print("Ready test matrix:", X_test_ready.shape)


## 8. Compare StandardScaler and MinMaxScaler
The comparison is made using training data only. StandardScaler typically works well for linear models when features have different units; MinMaxScaler maps each numeric feature into a bounded interval, but can compress most observations when extreme values are present. The choice here is StandardScaler, while the comparison illustrates the alternative without changing the held-out test data.


In [ ]:
if numeric_features:
    numeric_train = X_train[numeric_features].copy()
    num_imputer = SimpleImputer(strategy="median")
    numeric_train_imp = num_imputer.fit_transform(numeric_train)

    standard_scaled = StandardScaler().fit_transform(numeric_train_imp)
    minmax_scaled = MinMaxScaler().fit_transform(numeric_train_imp)
    scaler_comparison = pd.DataFrame({
        "scaler": ["StandardScaler", "MinMaxScaler"],
        "example_feature_min_after_scaling": [standard_scaled.min(), minmax_scaled.min()],
        "example_feature_max_after_scaling": [standard_scaled.max(), minmax_scaled.max()],
        "note": [
            "Centers each feature around 0 and scales to unit variance.",
            "Maps training feature ranges to [0, 1]; sensitive to extreme values."
        ]
    })
    display(scaler_comparison)
else:
    print("No numeric predictor columns; scaling comparison is not applicable.")


## 9. Train Linear Regression baseline and evaluate
A log-transformed target is used to moderate the heavy right tail. Metrics are calculated after converting predictions back to `CopiesSold`, so RMSE and MAE are in copies sold. Negative predictions are clipped to zero because negative sales are not meaningful. R² can be negative when the model performs worse than a constant-mean baseline.


In [ ]:
model = LinearRegression()
y_train_log = np.log1p(y_train)
model.fit(X_train_ready, y_train_log)

pred_log = model.predict(X_test_ready)
y_pred = np.maximum(0, np.expm1(pred_log))
y_true = y_test.to_numpy()

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

metrics = pd.DataFrame({
    "metric": ["RMSE", "MAE", "R²"],
    "value": [rmse, mae, r2],
    "interpretation": [
        "Typical error scale, penalizes large misses; measured in copies sold.",
        "Average absolute prediction error; measured in copies sold.",
        "Variance explained on the held-out test set; may be negative."
    ]
})
display(metrics)


## 10. Feature analysis
For one-hot encoded predictors, coefficients show the direction of the model's conditional association on the log-sales target scale. Because categorical levels create multiple coefficients and correlated features can make coefficients unstable, interpret this as a baseline diagnostic rather than causal importance.


In [ ]:
coef_df = pd.DataFrame({
    "feature": X_train_ready.columns,
    "coefficient": model.coef_,
    "abs_coefficient": np.abs(model.coef_)
}).sort_values("abs_coefficient", ascending=False)

print("Top 20 features by absolute linear-regression coefficient:")
display(coef_df.head(20))

top = coef_df.head(15).sort_values("abs_coefficient")
if not top.empty:
    plt.figure(figsize=(10, 6))
    plt.barh(top["feature"], top["coefficient"])
    plt.title("Largest Linear Regression Coefficients (log CopiesSold target)")
    plt.xlabel("Coefficient")
    plt.tight_layout()
    plt.show()


## 11. Prediction error analysis
Inspect the largest absolute errors, the actual-versus-predicted relationship, and residual distribution. For a sales dataset, a small number of very popular games can dominate RMSE; compare absolute and relative errors and be cautious with relative errors for small targets.


In [ ]:
error_df = pd.DataFrame({
    "actual_CopiesSold": y_true,
    "predicted_CopiesSold": y_pred,
    "absolute_error": np.abs(y_true - y_pred),
    "signed_error": y_pred - y_true
}, index=y_test.index).sort_values("absolute_error", ascending=False)

display(error_df.head(10))

plt.figure(figsize=(7, 6))
plt.scatter(y_true, y_pred, alpha=0.6)
limit = max(float(np.max(y_true)), float(np.max(y_pred)))
plt.plot([0, limit], [0, limit], linestyle="--")
plt.xlabel("Actual CopiesSold")
plt.ylabel("Predicted CopiesSold")
plt.title("Actual vs Predicted CopiesSold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
residuals = y_true - y_pred
plt.hist(residuals, bins=30)
plt.xlabel("Residual (actual - predicted)")
plt.ylabel("Count")
plt.title("Test Residual Distribution")
plt.tight_layout()
plt.show()

print("Mean signed error (prediction - actual):", float(np.mean(y_pred - y_true)))
print("Median absolute error:", float(np.median(np.abs(y_true - y_pred))))


## 12. Save deliverables
This saves the cleaned rows and the encoded training feature matrix for reproducibility. The fitted preprocessing pipeline and model can be saved with joblib. The encoded train matrix is the matrix displayed before training; it contains only training rows and avoids leaking test statistics into preprocessing.


In [ ]:
import joblib

df.to_csv("indie_games_cleaned.csv", index=False)
X_train_ready.to_csv("indie_games_X_train_ready.csv", index=False)
X_test_ready.to_csv("indie_games_X_test_ready.csv", index=False)
joblib.dump({"preprocessor": preprocessor, "model": model}, "indie_games_linear_regression_pipeline.joblib")
metrics.to_csv("model_metrics.csv", index=False)

print("Saved: indie_games_cleaned.csv")
print("Saved: indie_games_X_train_ready.csv")
print("Saved: indie_games_X_test_ready.csv")
print("Saved: indie_games_linear_regression_pipeline.joblib")
print("Saved: model_metrics.csv")
